In [1]:
import pandas as pd

# Load the file
df = pd.read_csv('baby-bonus-approved-institutions-ai.csv')

# Inspect data
print("--- HEAD ---")
print(df.head())
print("\n--- INFO ---")
print(df.info())

--- HEAD ---
         seq_no  instn_code                      instn_name  \
0  201500000373      PT9160   SAFARI HOUSE PRESCHOOL YISHUN   
1  201500000372     15C0226    TSI & PARTNERS FAMILY CLINIC   
2  201500000366  SS20150009  UNITED EYECARE (BOON KENG) LLP   
3  201500000364      YM0160         MY WORLD PRESCHOOL LTD.   
4  201500000363      YM0159         MY WORLD PRESCHOOL LTD.   

  instn_srvc_type_cde         instn_srvc_type instn_blk_hse_no  \
0                  CC        Childcare Centre                5   
1                  HC  Healthcare Institution              151   
2                  OS            Optical Shop               27   
3                  CC        Childcare Centre             787B   
4                  CC        Childcare Centre              270   

            instn_street_name instn_flr_no instn_unit_no  \
0  YISHUN INDUSTRIAL STREET 1           01            02   
1    SERANGOON NORTH AVENUE 2           01            01   
2              BENDEMEER ROAD  

In [2]:
# Let's explore seq_no and service types
print(df['instn_srvc_type'].value_counts())
print("\nSample seq_no values:")
print(df['seq_no'].describe())
print(df['seq_no'].head(10))
print(df['seq_no'].tail(10))

instn_srvc_type
Childcare Centre                         1549
Healthcare Institution                   1505
Optical Shop                              692
Kindergarten                              442
Pharmacy                                  229
Early Intervention Programmes             131
Special Education                          23
Assistive Technology Device Providers      21
CPE Kindergarten                            8
Insurance                                   1
Name: count, dtype: int64

Sample seq_no values:
count    4.601000e+03
mean     1.816963e+11
std      5.939068e+10
min      2.001000e+09
25%      2.009000e+11
50%      2.013000e+11
75%      2.016000e+11
max      2.020000e+11
Name: seq_no, dtype: float64
0    201500000373
1    201500000372
2    201500000366
3    201500000364
4    201500000363
5    201500000362
6    201500000361
7    201500000360
8    201500000282
9    201500000280
Name: seq_no, dtype: int64
4591    202000000140
4592    202000000144
4593    202000000007


In [3]:
# Convert seq_no to string and inspect first 4 characters
years = df['seq_no'].astype(str).str[:4]
print(years.value_counts().sort_index())
df['approval_year'] = years.astype(int)
print(df.groupby(['approval_year', 'instn_srvc_type']).size().unstack(fill_value=0))

seq_no
2001    344
2002     38
2003     60
2004     62
2005     59
2006     41
2007    310
2008     87
2009    190
2010    189
2011    139
2012    632
2013    307
2014    377
2015    381
2016    316
2017    377
2018    357
2019    253
2020     82
Name: count, dtype: int64
instn_srvc_type  Assistive Technology Device Providers  CPE Kindergarten  \
approval_year                                                              
2001                                                 0                 0   
2002                                                 0                 0   
2003                                                 0                 0   
2004                                                 0                 0   
2005                                                 0                 0   
2006                                                 0                 0   
2007                                                 0                 1   
2008                                       

In [4]:
# Check postal code format
print(df['instn_pc'].head())
# Extract first 2 digits of postal code (Singapore postal sectors)
df['postal_sector'] = df['instn_pc'].astype(str).str.zfill(6).str[:2]
print("\nTop 10 postal sectors:")
print(df['postal_sector'].value_counts().head(10))

0    768161
1    550151
2    330027
3    732787
4    760270
Name: instn_pc, dtype: int64

Top 10 postal sectors:
postal_sector
52    258
53    210
76    199
73    197
64    190
56    175
54    165
46    157
82    145
68    141
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Re-load data to ensure consistency
df = pd.read_csv('baby-bonus-approved-institutions-ai.csv')
df['approval_year'] = df['seq_no'].astype(str).str[:4].astype(int)
df['postal_sector'] = df['instn_pc'].astype(str).str.zfill(6).str[:2]

# 1. Trend Analysis: Annual approvals by service type
trend_df = df.groupby(['approval_year', 'instn_srvc_type']).size().unstack(fill_value=0)
print("Trend summary (last 5 years):")
print(trend_df.tail())

# 2. Volume Analysis: Volume by postal sector and type
volume_sector = df.groupby(['postal_sector', 'instn_srvc_type']).size().unstack(fill_value=0)

# 3. Statistical Association (Categorical Correlation)
# Let's see the association between top postal sectors and top service types
top_sectors = df['postal_sector'].value_counts().head(10).index
top_types = df['instn_srvc_type'].value_counts().head(5).index

contingency_table = df[df['postal_sector'].isin(top_sectors) & df['instn_srvc_type'].isin(top_types)]
ct = pd.crosstab(contingency_table['postal_sector'], contingency_table['instn_srvc_type'])

chi2, p, dof, expected = chi2_contingency(ct)
print(f"\nChi-square statistic: {chi2}, p-value: {p}")

# 4. Distribution Analysis: Distribution of annual additions
annual_totals = df['approval_year'].value_counts().sort_index()
print("\nAnnual additions distribution summary:")
print(annual_totals.describe())

# Save transformed data for Power BI
# We can create a consolidated flattened table for Power BI modeling
df['postal_sector_full'] = 'Sector ' + df['postal_sector']
df.to_csv('transformed_baby_bonus_institutions.csv', index=False)
print("\nTransformed data saved to transformed_baby_bonus_institutions.csv")

Trend summary (last 5 years):
instn_srvc_type  Assistive Technology Device Providers  CPE Kindergarten  \
approval_year                                                              
2016                                                 0                 2   
2017                                                 1                 1   
2018                                                 0                 0   
2019                                                 0                 0   
2020                                                 0                 0   

instn_srvc_type  Childcare Centre  Early Intervention Programmes  \
approval_year                                                      
2016                          118                              8   
2017                          126                             19   
2018                          125                              9   
2019                           83                              7   
2020                         

In [6]:
# Create aggregated tables for Power BI

# 1. Annual trend summary
annual_trend = df.groupby(['approval_year', 'instn_srvc_type']).size().unstack(fill_value=0)
annual_trend['Total_Approvals'] = annual_trend.sum(axis=1)
annual_trend.to_csv('powerbi_annual_trend.csv')

# 2. Geographic sector volume summary
geo_volume = df.groupby(['postal_sector', 'instn_srvc_type']).size().unstack(fill_value=0)
geo_volume['Total_Institutions'] = geo_volume.sum(axis=1)
geo_volume.to_csv('powerbi_geographic_volume.csv')

# 3. Normalized Crosstab for correlation-like heatmap visualization
# We can find the percentage distribution of institution types across postal sectors
norm_crosstab = pd.crosstab(df['postal_sector'], df['instn_srvc_type'], normalize='index') * 100
norm_crosstab.to_csv('powerbi_correlation_heatmap.csv')

print("Created powerbi_annual_trend.csv, powerbi_geographic_volume.csv, and powerbi_correlation_heatmap.csv")

Created powerbi_annual_trend.csv, powerbi_geographic_volume.csv, and powerbi_correlation_heatmap.csv
